<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/RELATION_AUDITED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#DOWNLOADING THE DATASET ACS19955
from google.colab import drive
import requests
import os
from tqdm.auto import tqdm
drive.mount("/content/drive")
# ============================================================
# LABEL RELATIONSHIP AUDIT
# CELL 1 — LOAD FINAL TRAINING METADATA
# ============================================================

import os
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/ECG_Project"

SPLIT_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "dataset_split"
)

PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "preprocessing",
    "processed"
)

TRAIN_CSV = os.path.join(
    SPLIT_DIR,
    "train_split.csv"
)

TRAIN_H5 = os.path.join(
    PROCESSED_DIR,
    "train_fixed.h5"
)

print("=" * 70)
print("LABEL RELATIONSHIP AUDIT")
print("=" * 70)

print("\nLoading training metadata...")

train_df = pd.read_csv(TRAIN_CSV)

print("Training CSV shape:", train_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nFirst 5 rows:")
display(train_df.head())

Mounted at /content/drive
LABEL RELATIONSHIP AUDIT

Loading training metadata...
Training CSV shape: (15275, 28)

Columns:
['Patient_id', 'ecg_row_record', 'ecg_med_record', 'gender', 'age', 'Time_Interval', 'AMI', 'OMI', 'CTO', 'PCI', 'NSTEMI', 'STEMI', 'UA', 'LM', 'PLAD', 'MLAD', 'DLAD', 'DB', 'PLCX', 'MLCX', 'DLCX', 'OM', 'PRCA', 'MRCA', 'DRCA', 'VF_VT', 'Paced', 'Prior_PCI']

First 5 rows:


,Patient_id,ecg_row_record,ecg_med_record,gender,age,Time_Interval,AMI,OMI,CTO,PCI,...,PLCX,MLCX,DLCX,OM,PRCA,MRCA,DRCA,VF_VT,Paced,Prior_PCI
0,P00001,04904.dat,04904.med,1,58,67,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,P00002,15878.dat,15878.med,1,68,2895,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
2,P00002,13502.dat,13502.med,1,67,1633,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,P00003,02934.dat,02934.med,1,57,440,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,P00005,15847.dat,15847.med,1,28,206,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ============================================================
# CELL 2 — ALIGN FINAL H5 RECORDS WITH TRAINING METADATA
# ============================================================

import h5py

with h5py.File(TRAIN_H5, "r") as h5:
    h5_ids = [
        x.decode() if isinstance(x, bytes) else str(x)
        for x in h5["record_ids"][:]
    ]

h5_ids = pd.Series(h5_ids, name="record_id")

print("Final H5 ECGs :", len(h5_ids))
print("Training CSV  :", len(train_df))

# ------------------------------------------------------------
# Normalize record IDs
# ------------------------------------------------------------

train_df["record_id"] = (
    train_df["ecg_row_record"]
    .astype(str)
    .str.replace(".dat", "", regex=False)
    .str.zfill(5)
)

h5_id_set = set(h5_ids)

# Keep only ECGs that actually exist in final H5
train_final = train_df[
    train_df["record_id"].isin(h5_id_set)
].copy()

# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

print("\nAfter alignment:")
print("Usable training ECGs:", len(train_final))

print("\nMissing from final H5:")

missing = sorted(
    set(train_df["record_id"]) - h5_id_set
)

print(missing)

print("\nExtra H5 IDs:")

extra = sorted(
    h5_id_set - set(train_df["record_id"])
)

print(extra)

assert len(train_final) == len(h5_ids)

print("\n✓ CSV ↔ H5 alignment verified")

Final H5 ECGs : 15273
Training CSV  : 15275

After alignment:
Usable training ECGs: 15273

Missing from final H5:
['03228', '14262']

Extra H5 IDs:
[]

✓ CSV ↔ H5 alignment verified


In [ ]:
# ============================================================
# CELL 3 — IDENTIFY CLINICAL LABEL COLUMNS
# ============================================================

print("=" * 70)
print("IDENTIFYING LABEL COLUMNS")
print("=" * 70)

for i, col in enumerate(train_final.columns):
    print(f"{i:2d} : {col}")

IDENTIFYING LABEL COLUMNS
 0 : Patient_id
 1 : ecg_row_record
 2 : ecg_med_record
 3 : gender
 4 : age
 5 : Time_Interval
 6 : AMI
 7 : OMI
 8 : CTO
 9 : PCI
10 : NSTEMI
11 : STEMI
12 : UA
13 : LM
14 : PLAD
15 : MLAD
16 : DLAD
17 : DB
18 : PLCX
19 : MLCX
20 : DLCX
21 : OM
22 : PRCA
23 : MRCA
24 : DRCA
25 : VF_VT
26 : Paced
27 : Prior_PCI
28 : record_id


In [ ]:
# ============================================================
# CELL 4 — DEFINE CLINICAL LABELS
# ============================================================

# IMPORTANT:
# Replace/add names here ONLY after inspecting Cell 3.

EXCLUDE_COLUMNS = [
    "Patient_id",
    "ecg_row_record",
    "ecg_med_record",
    "gender",
    "age",
    "Time_Interval",
    "record_id"
]

LABEL_COLUMNS = [
    col for col in train_final.columns
    if col not in EXCLUDE_COLUMNS
]

print("=" * 70)
print("CLINICAL LABEL COLUMNS")
print("=" * 70)

print("Number of labels:", len(LABEL_COLUMNS))

for i, label in enumerate(LABEL_COLUMNS, 1):
    print(f"{i:2d}. {label}")

assert len(LABEL_COLUMNS) == 22, (
    f"Expected 22 labels, found {len(LABEL_COLUMNS)}"
)

print("\n✓ 22 clinical label columns identified")

CLINICAL LABEL COLUMNS
Number of labels: 22
 1. AMI
 2. OMI
 3. CTO
 4. PCI
 5. NSTEMI
 6. STEMI
 7. UA
 8. LM
 9. PLAD
10. MLAD
11. DLAD
12. DB
13. PLCX
14. MLCX
15. DLCX
16. OM
17. PRCA
18. MRCA
19. DRCA
20. VF_VT
21. Paced
22. Prior_PCI

✓ 22 clinical label columns identified


In [ ]:
# ============================================================
# CELL 5 — CLASS DISTRIBUTION
# ============================================================

print("=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

distribution = []

for label in LABEL_COLUMNS:

    counts = train_final[label].value_counts(
        dropna=False
    )

    total = len(train_final)

    distribution.append({
        "label": label,
        "class_0": counts.get(0, 0),
        "class_1": counts.get(1, 0),
        "class_2": counts.get(2, 0),
        "missing": train_final[label].isna().sum()
    })

distribution_df = pd.DataFrame(distribution)

display(distribution_df)

CLASS DISTRIBUTION


,label,class_0,class_1,class_2,missing
0,AMI,12978,2295,0,0
1,OMI,14288,985,0,0
2,CTO,14463,810,0,0
3,PCI,11424,3849,0,0
4,NSTEMI,14204,1069,0,0
5,STEMI,14049,1224,0,0
6,UA,9968,5305,0,0
7,LM,15265,4,4,0
8,PLAD,14803,241,229,0
9,MLAD,15082,104,87,0


In [ ]:
# ============================================================
# CELL 6 — LABEL TYPE AUDIT
# ============================================================

print("=" * 70)
print("LABEL TYPE AUDIT")
print("=" * 70)

for label in LABEL_COLUMNS:

    values = sorted(
        train_final[label].dropna().unique().tolist()
    )

    print(
        f"{label:15s} | "
        f"values = {values} | "
        f"classes = {len(values)}"
    )

LABEL TYPE AUDIT
AMI             | values = [0, 1] | classes = 2
OMI             | values = [0, 1] | classes = 2
CTO             | values = [0, 1] | classes = 2
PCI             | values = [0, 1] | classes = 2
NSTEMI          | values = [0, 1] | classes = 2
STEMI           | values = [0, 1] | classes = 2
UA              | values = [0, 1] | classes = 2
LM              | values = [0, 1, 2] | classes = 3
PLAD            | values = [0, 1, 2] | classes = 3
MLAD            | values = [0, 1, 2] | classes = 3
DLAD            | values = [0, 1, 2] | classes = 3
DB              | values = [0, 1, 2] | classes = 3
PLCX            | values = [0, 1, 2] | classes = 3
MLCX            | values = [0, 1, 2] | classes = 3
DLCX            | values = [0, 1, 2] | classes = 3
OM              | values = [0, 1, 2] | classes = 3
PRCA            | values = [0, 1, 2] | classes = 3
MRCA            | values = [0, 1, 2] | classes = 3
DRCA            | values = [0, 1, 2] | classes = 3
VF_VT           | values = [0, 1] |

In [ ]:
# ============================================================
# CELL 7 — LABEL CORRELATION
# ============================================================

print("=" * 70)
print("LABEL CORRELATION MATRIX")
print("=" * 70)

correlation = train_final[
    LABEL_COLUMNS
].corr()

display(
    correlation.round(3)
)

LABEL CORRELATION MATRIX


,AMI,OMI,CTO,PCI,NSTEMI,STEMI,UA,LM,PLAD,MLAD,...,PLCX,MLCX,DLCX,OM,PRCA,MRCA,DRCA,VF_VT,Paced,Prior_PCI
AMI,1.000,0.624,-0.100,0.265,0.652,0.702,-0.307,0.011,0.112,0.082,...,0.084,0.073,0.046,0.009,0.117,0.081,0.089,0.018,-0.025,-0.005
OMI,0.624,1.000,-0.062,0.194,0.300,0.539,-0.192,0.024,0.215,0.151,...,0.161,0.132,0.096,0.027,0.219,0.159,0.161,0.023,-0.009,-0.003
CTO,-0.100,-0.062,1.000,0.150,-0.065,-0.070,0.191,0.059,0.367,0.231,...,0.268,0.192,0.240,0.121,0.336,0.296,0.248,-0.005,-0.016,0.050
PCI,0.265,0.194,0.150,1.000,0.152,0.206,0.201,0.029,0.113,0.064,...,0.070,0.061,0.077,0.029,0.108,0.075,0.091,-0.020,-0.038,0.343
NSTEMI,0.652,0.300,-0.065,0.152,1.000,-0.081,-0.200,0.001,0.022,0.032,...,0.074,0.071,0.029,-0.000,0.047,0.032,0.052,-0.008,-0.014,0.002
STEMI,0.702,0.539,-0.070,0.206,-0.081,1.000,-0.215,0.014,0.126,0.077,...,0.042,0.030,0.033,0.012,0.107,0.077,0.068,0.032,-0.020,-0.009
UA,-0.307,-0.192,0.191,0.201,-0.200,-0.215,1.000,-0.008,0.020,0.011,...,0.013,0.008,0.029,0.009,0.024,0.025,0.017,-0.074,-0.033,0.149
LM,0.011,0.024,0.059,0.029,0.001,0.014,-0.008,1.000,-0.004,-0.002,...,-0.002,-0.002,-0.002,-0.001,-0.003,-0.003,-0.002,-0.005,-0.003,-0.006
PLAD,0.112,0.215,0.367,0.113,0.022,0.126,0.020,-0.004,1.000,-0.018,...,0.029,0.010,0.019,0.027,0.075,0.020,0.046,-0.000,-0.004,0.014
MLAD,0.082,0.151,0.231,0.064,0.032,0.077,0.011,-0.002,-0.018,1.000,...,0.061,0.064,0.062,0.023,0.044,0.023,0.003,0.003,-0.012,-0.001


In [ ]:
# ============================================================
# CELL 8 — PAIRWISE CO-OCCURRENCE
# ============================================================

binary_labels = []

for label in LABEL_COLUMNS:

    values = set(
        train_final[label].dropna().unique()
    )

    if values.issubset({0, 1}):
        binary_labels.append(label)

print("Binary labels:")
print(binary_labels)

cooccurrence = pd.DataFrame(
    index=binary_labels,
    columns=binary_labels,
    dtype=int
)

for label_a in binary_labels:

    for label_b in binary_labels:

        cooccurrence.loc[label_a, label_b] = (
            (
                (train_final[label_a] == 1) &
                (train_final[label_b] == 1)
            ).sum()
        )

print("\nNumber of ECGs where both labels are positive:")

display(cooccurrence)

Binary labels:
['AMI', 'OMI', 'CTO', 'PCI', 'NSTEMI', 'STEMI', 'UA', 'VF_VT', 'Paced', 'Prior_PCI']

Number of ECGs where both labels are positive:


,AMI,OMI,CTO,PCI,NSTEMI,STEMI,UA,VF_VT,Paced,Prior_PCI
AMI,2295.0,985.0,0.0,1207.0,1069.0,1224.0,0.0,130.0,25.0,162.0
OMI,985.0,985.0,0.0,564.0,356.0,628.0,0.0,65.0,14.0,70.0
CTO,0.0,0.0,810.0,427.0,0.0,0.0,592.0,35.0,8.0,105.0
PCI,1207.0,564.0,427.0,3849.0,526.0,680.0,1973.0,155.0,38.0,879.0
NSTEMI,1069.0,356.0,0.0,526.0,1069.0,0.0,0.0,44.0,13.0,81.0
STEMI,1224.0,628.0,0.0,680.0,0.0,1224.0,0.0,86.0,12.0,81.0
UA,0.0,0.0,592.0,1973.0,0.0,0.0,5305.0,138.0,67.0,676.0
VF_VT,130.0,65.0,35.0,155.0,44.0,86.0,138.0,725.0,42.0,42.0
Paced,25.0,14.0,8.0,38.0,13.0,12.0,67.0,42.0,288.0,16.0
Prior_PCI,162.0,70.0,105.0,879.0,81.0,81.0,676.0,42.0,16.0,1129.0


In [ ]:
# ============================================================
# CELL 9 — CONDITIONAL PROBABILITIES
# ============================================================

print("=" * 70)
print("CONDITIONAL RELATIONSHIPS")
print("=" * 70)

conditional_rows = []

for source in binary_labels:

    source_positive = (
        train_final[source] == 1
    )

    source_count = source_positive.sum()

    if source_count == 0:
        continue

    for target in binary_labels:

        if source == target:
            continue

        both = (
            source_positive &
            (train_final[target] == 1)
        ).sum()

        probability = both / source_count

        conditional_rows.append({
            "condition": source,
            "target": target,
            "condition_count": source_count,
            "both_positive": both,
            "P(target=1 | condition=1)": probability
        })

conditional_df = pd.DataFrame(
    conditional_rows
)

conditional_df = conditional_df.sort_values(
    "P(target=1 | condition=1)",
    ascending=False
)

display(
    conditional_df.head(50)
)

CONDITIONAL RELATIONSHIPS


,condition,target,condition_count,both_positive,P(target=1 | condition=1)
9,OMI,AMI,985,985,1.000000
36,NSTEMI,AMI,1069,1069,1.000000
45,STEMI,AMI,1224,1224,1.000000
84,Prior_PCI,PCI,1129,879,0.778565
23,CTO,UA,810,592,0.730864
13,OMI,STEMI,985,628,0.637563
87,Prior_PCI,UA,1129,676,0.598760
11,OMI,PCI,985,564,0.572589
48,STEMI,PCI,1224,680,0.555556
4,AMI,STEMI,2295,1224,0.533333


In [ ]:
# ============================================================
# CELL 10 — OMI ↔ CORONARY VESSEL RELATIONSHIP
# ============================================================

VESSEL_LABELS = [
    "PLAD",
    "MLAD",
    "DLAD",
    "PLCX",
    "MLCX",
    "DLCX",
    "PRCA",
    "MRCA",
    "DRCA"
]

available_vessels = [
    x for x in VESSEL_LABELS
    if x in train_final.columns
]

print("Available vessel labels:")
print(available_vessels)

for vessel in available_vessels:

    print("\n" + "-" * 60)
    print("VESSEL:", vessel)

    for state in [0, 1, 2]:

        subset = train_final[
            train_final[vessel] == state
        ]

        if len(subset) == 0:
            continue

        omi_rate = (
            subset["OMI"].mean()
            if "OMI" in subset.columns
            else np.nan
        )

        print(
            f"{vessel} = {state}: "
            f"N={len(subset):5d} | "
            f"OMI rate={omi_rate:.4f}"
        )

Available vessel labels:
['PLAD', 'MLAD', 'DLAD', 'PLCX', 'MLCX', 'DLCX', 'PRCA', 'MRCA', 'DRCA']

------------------------------------------------------------
VESSEL: PLAD
PLAD = 0: N=14803 | OMI rate=0.0495
PLAD = 1: N=  241 | OMI rate=0.9959
PLAD = 2: N=  229 | OMI rate=0.0524

------------------------------------------------------------
VESSEL: MLAD
MLAD = 0: N=15082 | OMI rate=0.0581
MLAD = 1: N=  104 | OMI rate=0.9904
MLAD = 2: N=   87 | OMI rate=0.0690

------------------------------------------------------------
VESSEL: DLAD
DLAD = 0: N=15250 | OMI rate=0.0639
DLAD = 1: N=   10 | OMI rate=1.0000
DLAD = 2: N=   13 | OMI rate=0.0769

------------------------------------------------------------
VESSEL: PLCX
PLCX = 0: N=15052 | OMI rate=0.0573
PLCX = 1: N=  112 | OMI rate=0.9911
PLCX = 2: N=  109 | OMI rate=0.1101

------------------------------------------------------------
VESSEL: MLCX
MLCX = 0: N=15136 | OMI rate=0.0599
MLCX = 1: N=   74 | OMI rate=0.9865
MLCX = 2: N=   63 | OMI

In [ ]:
# ============================================================
# CELL 11 — OMI / CTO / VESSEL RELATIONSHIP
# ============================================================

for vessel in available_vessels:

    print("\n" + "=" * 70)
    print("VESSEL:", vessel)
    print("=" * 70)

    table = pd.crosstab(
        [
            train_final["OMI"],
            train_final["CTO"]
        ],
        train_final[vessel]
    )

    print(table)


VESSEL: PLAD
PLAD         0    1    2
OMI CTO                 
0   0    13449    1   28
    1      621    0  189
1   0      733  240   12

VESSEL: MLAD
MLAD         0    1   2
OMI CTO                
0   0    13471    1   6
    1      735    0  75
1   0      876  103   6

VESSEL: DLAD
DLAD         0   1   2
OMI CTO               
0   0    13477   0   1
    1      799   0  11
1   0      974  10   1

VESSEL: PLCX
PLCX         0    1   2
OMI CTO                
0   0    13475    1   2
    1      715    0  95
1   0      862  111  12

VESSEL: MLCX
MLCX         0   1   2
OMI CTO               
0   0    13473   1   4
    1      757   0  53
1   0      906  73   6

VESSEL: DLCX
DLCX         0   1   2
OMI CTO               
0   0    13476   1   1
    1      741   0  69
1   0      924  53   8

VESSEL: PRCA
PRCA         0    1    2
OMI CTO                 
0   0    13461    3   14
    1      644    0  166
1   0      773  171   41

VESSEL: MRCA
MRCA         0    1    2
OMI CTO                 
0  

In [ ]:
# ============================================================
# CELL 12 — ACS RELATIONSHIP
# ============================================================

ACS_LABELS = [
    "STEMI",
    "NSTEMI",
    "UA"
]

available_acs = [
    x for x in ACS_LABELS
    if x in train_final.columns
]

print("ACS labels found:")
print(available_acs)

print("\nACS combinations:")

acs_table = train_final[
    available_acs
].value_counts().reset_index(name="count")

display(acs_table)

ACS labels found:
['STEMI', 'NSTEMI', 'UA']

ACS combinations:


,STEMI,NSTEMI,UA,count
0,0,0,0,7675
1,0,0,1,5305
2,1,0,0,1224
3,0,1,0,1069


In [ ]:
# ============================================================
# CELL 13 — CLINICAL CONSISTENCY CHECKS
# ============================================================

print("=" * 70)
print("CLINICAL CONSISTENCY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# ACS labels
# ------------------------------------------------------------

if set(["STEMI", "NSTEMI", "UA"]).issubset(
    train_final.columns
):

    acs_sum = train_final[
        ["STEMI", "NSTEMI", "UA"]
    ].sum(axis=1)

    print("\nACS label combinations:")

    print(
        acs_sum.value_counts().sort_index()
    )

    print(
        "\nPatients with >1 ACS category:",
        (acs_sum > 1).sum()
    )

    print(
        "Patients with no ACS category:",
        (acs_sum == 0).sum()
    )


# ------------------------------------------------------------
# OMI / CTO
# ------------------------------------------------------------

if set(["OMI", "CTO"]).issubset(
    train_final.columns
):

    print("\nOMI / CTO combinations:")

    print(
        pd.crosstab(
            train_final["OMI"],
            train_final["CTO"]
        )
    )

CLINICAL CONSISTENCY CHECK

ACS label combinations:
0    7675
1    7598
Name: count, dtype: int64

Patients with >1 ACS category: 0
Patients with no ACS category: 7675

OMI / CTO combinations:
CTO      0    1
OMI            
0    13478  810
1      985    0


In [ ]:
# ============================================================
# CELL 14 — SAVE AUDIT RESULTS
# ============================================================

AUDIT_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "label_audit"
)

os.makedirs(
    AUDIT_DIR,
    exist_ok=True
)

distribution_df.to_csv(
    os.path.join(
        AUDIT_DIR,
        "label_distribution.csv"
    ),
    index=False
)

correlation.to_csv(
    os.path.join(
        AUDIT_DIR,
        "label_correlation.csv"
    )
)

cooccurrence.to_csv(
    os.path.join(
        AUDIT_DIR,
        "binary_cooccurrence.csv"
    )
)

conditional_df.to_csv(
    os.path.join(
        AUDIT_DIR,
        "conditional_relationships.csv"
    ),
    index=False
)

print("=" * 70)
print("LABEL AUDIT RESULTS SAVED")
print("=" * 70)

print(AUDIT_DIR)

LABEL AUDIT RESULTS SAVED
/content/drive/MyDrive/ECG_Project/results/label_audit
